In [1]:
import pandas as pd
import glob

csv_files = glob.glob(
    "/kaggle/input/**/arabic_environment_texts.csv",
    recursive=True
)

print(csv_files)

['/kaggle/input/datasets/chikhimane/arabic-environment-texts/arabic_environment_texts.csv']


In [2]:
df = pd.read_csv(csv_files[0], encoding="utf-8-sig")

print("Dataset shape:", df.shape)
print(df.head())
print(df.info())

Dataset shape: (117, 1)
                                                text
0  حذر اتحاد بلديات قطاع غزة من انهيار وشيك في من...
1  وقال رئيس بلدية خان يونس الدكتور علاء الدين ال...
2  وحذر البطة من أن الأزمة قد تقود إلى عطش جماعي ...
3  وأكد الاتحاد أن البلديات لم تتلق -منذ مطلع مار...
4  ولا يقتصر أثر نقص الوقود والزيوت على مرافق الم...
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 117 entries, 0 to 116
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    117 non-null    object
dtypes: object(1)
memory usage: 1.0+ KB
None


In [3]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel
import numpy as np

import logging
logging.getLogger().setLevel(logging.CRITICAL)

import warnings
warnings.filterwarnings('ignore')

device = 'cpu'
if torch.cuda.is_available():
    device = 'cuda'

print(device)

cpu


In [4]:
!pip install -q transformers datasets accelerate

In [5]:
!pip install -q arabert

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from arabert.preprocess import ArabertPreprocessor

MODEL_NAME = "aubmindlab/aragpt2-base"

# Préprocesseur arabe
arabert_prep = ArabertPreprocessor(model_name=MODEL_NAME)

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Ajouter un token de padding distinct
tokenizer.add_special_tokens({"pad_token": "<|pad|>"})

# Modèle GPT-2
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.resize_token_embeddings(len(tokenizer))

# Utiliser le GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Device:", device)
print("Model:", MODEL_NAME)
print("Vocabulary size:", len(tokenizer))
print("Number of parameters:", sum(p.numel() for p in model.parameters()))

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.50M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.52M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  553MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: aubmindlab/aragpt2-base
Key                         | Status     |  | 
----------------------------+------------+--+-
h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Device: cpu
Model: aubmindlab/aragpt2-base
Vocabulary size: 64001
Number of parameters: 134995200


In [7]:
from datasets import Dataset
from transformers import DataCollatorForLanguageModeling

# 1. Garder uniquement les textes originaux
df_generation = df[["text"]].dropna().copy()
df_generation["text"] = df_generation["text"].astype(str).str.strip()
df_generation = df_generation[df_generation["text"] != ""]

# 2. Prétraiter les textes arabes
df_generation["text"] = df_generation["text"].apply(
    arabert_prep.preprocess
)

# 3. Convertir en Dataset Hugging Face
dataset = Dataset.from_pandas(
    df_generation,
    preserve_index=False
)

# 4. Séparer entraînement et validation
dataset = dataset.train_test_split(
    test_size=0.1,
    seed=42
)

# 5. Tokenisation
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=256
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

# 6. Préparer le Data Collator pour GPT-2
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print("Train examples:", len(tokenized_dataset["train"]))
print("Validation examples:", len(tokenized_dataset["test"]))
print("\nExample tokenized text:")
print(tokenizer.decode(
    tokenized_dataset["train"][0]["input_ids"]
))

Map:   0%|          | 0/105 [00:00<?, ? examples/s]

Map:   0%|          | 0/12 [00:00<?, ? examples/s]

Train examples: 105
Validation examples: 12

Example tokenized text:
وتكتسب المنطقة حساسية بيئية خاصة ؛ إذ تضم جزر الحلانيات مواقع للشعاب المرجانية ، وتعد المياه العمانية موطنا لمجموعة الحوت الأحدب في بحر العرب ، التي تصفها هيئة البيئة بأنها مجموعة فريدة وراثيا ومهددة بالانقراض ، ومن أندر مجموعات الحيتان البالينية في العالم ، كما تضم المنطقة طيورا بحرية بينها غراب البحر السقطري .


In [8]:
#refaire la tokenisation
def tokenize_function(examples):
    all_input_ids = []

    for text in examples["text"]:
        ids = tokenizer(
            text,
            truncation=True,
            max_length=255,
            add_special_tokens=False
        )["input_ids"]

        ids.append(tokenizer.eos_token_id)
        all_input_ids.append(ids)

    return {
        "input_ids": all_input_ids,
        "attention_mask": [
            [1] * len(ids) for ids in all_input_ids
        ]
    }

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

print("Train:", len(tokenized_dataset["train"]))
print("Validation:", len(tokenized_dataset["test"]))

Map:   0%|          | 0/105 [00:00<?, ? examples/s]

Map:   0%|          | 0/12 [00:00<?, ? examples/s]

Train: 105
Validation: 12


In [9]:
#Définir les paramètres et entraîner GPT-2
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./aragpt2-environment",

    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,

    learning_rate=5e-5,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=1,

    logging_steps=5,
    fp16=True,

    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
    processing_class=tokenizer
)

print("Training configuration ready!")

Training configuration ready!


In [10]:
train_result = trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 64000}.
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,7.371965,3.907503
2,6.219226,3.929553
3,5.812547,3.904632


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


In [11]:
# Utiliser le meilleur checkpoint
model = trainer.model
model.eval()

# Prompt en arabe
prompt = "يعد التلوث البيئي من أخطر المشكلات التي تواجه العالم اليوم،"

# Appliquer le même prétraitement
prompt = arabert_prep.preprocess(prompt)

# Tokenisation du prompt
inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(device)

# Génération
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.8,
        top_k=50,
        top_p=0.9,
        repetition_penalty=1.2,
        no_repeat_ngram_size=3,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

# Décoder uniquement le texte généré
generated_text = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("PROMPT:")
print(prompt)

print("\nGENERATED TEXT:")
print(generated_text)

PROMPT:
يعد التلوث البيئي من أخطر المشكلات التي تواجه العالم اليوم ،

GENERATED TEXT:
 وقد تسبب التلوث البيئي الناجم عن تلوث الهواء والتربة والمياه والهواء في المناطق الملوثة .


In [12]:
prompts = [
    "تعتبر مشكلة تلوث المياه من أهم التحديات البيئية،",
    "تؤثر النفايات البلاستيكية على الحياة البحرية،",
    "تساهم الغازات الناتجة عن المصانع في تلوث الهواء،"
]

model.eval()

for prompt in prompts:
    processed_prompt = arabert_prep.preprocess(prompt)

    inputs = tokenizer(
        processed_prompt,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=True,
            temperature=0.8,
            top_k=50,
            top_p=0.9,
            repetition_penalty=1.2,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    print("\nPROMPT:", prompt)
    print("GENERATED:", generated)
    print("-" * 80)


PROMPT: تعتبر مشكلة تلوث المياه من أهم التحديات البيئية،
GENERATED:  حيث يتم إعادة معالجة مياه الصرف الصحي على نحو 40 % من مياه البحر المتوسط سنويا ، بالإضافة إلى أن هذه المشكلة تؤثر بشكل مباشر في المناطق الساحلية المحيطة بالمنطقة .
--------------------------------------------------------------------------------

PROMPT: تؤثر النفايات البلاستيكية على الحياة البحرية،
GENERATED:  وتصيب الإنسان والحيوان .
--------------------------------------------------------------------------------

PROMPT: تساهم الغازات الناتجة عن المصانع في تلوث الهواء،
GENERATED:  بما يؤدي إلى زيادة حجم الملوثات العضوية والمواد الكيميائية ، وتؤثر سلبا على المواد الغذائية والمياه والهواء .
--------------------------------------------------------------------------------


In [13]:
def generate_paragraph(prompt, max_new_tokens=150):

    # 1. Prétraitement du texte arabe
    prompt = arabert_prep.preprocess(prompt)

    # 2. Tokenisation
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(device)

    # 3. Génération
    model.eval()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.8,
            top_k=50,
            top_p=0.9,
            repetition_penalty=1.2,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    # 4. Décodage : récupérer uniquement la continuation
    generated_text = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return generated_text.strip()

In [14]:
user_sentence = "تعتبر الطاقة المتجددة من الحلول المهمة لحماية البيئة،"

new_paragraph = generate_paragraph(
    user_sentence,
    max_new_tokens=150
)

print("Input sentence:")
print(user_sentence)

print("\nGenerated paragraph:")
print(new_paragraph)

Input sentence:
تعتبر الطاقة المتجددة من الحلول المهمة لحماية البيئة،

Generated paragraph:
وهي تشمل في الواقع حلولا مستدامة على المدى الطويل ، إذ تعتمد مصادر الطاقة البديلة على توليد الطاقة الكهربائية والمياه ، فيما تعد النفايات الصلبة إحدى أهم التحديات التي تواجه الجهود المحلية .


In [15]:
test_sentences = [
    "تعتبر الطاقة المتجددة من الحلول المهمة لحماية البيئة،",
    "يشكل تلوث الهواء خطرا كبيرا على صحة الإنسان،",
    "تساهم إعادة تدوير النفايات في الحفاظ على الموارد الطبيعية،"
]

for sentence in test_sentences:

    generated = generate_paragraph(sentence)

    print("\n" + "=" * 80)
    print("INPUT:", sentence)
    print("OUTPUT:", generated)


INPUT: تعتبر الطاقة المتجددة من الحلول المهمة لحماية البيئة،
OUTPUT: حيث أن هذه التحديات تتطلب إعادة تنظيم وإدارة الطاقة المحلية .

INPUT: يشكل تلوث الهواء خطرا كبيرا على صحة الإنسان،
OUTPUT: إذ أظهرت البيانات أن 70 % من الأشخاص يعيشون في المناطق الصناعية الخطرة وغير الآمنة .

INPUT: تساهم إعادة تدوير النفايات في الحفاظ على الموارد الطبيعية،
OUTPUT: إذ تم إنشاء العديد من الطرق الزراعية في الأراضي المحيطة بالحيوانات .


In [16]:
model.save_pretrained("./aragpt2-environment-final")
tokenizer.save_pretrained("./aragpt2-environment-final")

print("Model and tokenizer saved successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved successfully!
